# 09 · Constrained policy allocation (SciPy + HiGHS)
Maximize **model-projected** incremental NCM under the budget, offer cap, PCT_20 share among commercial recipients,
and a pre-reserved randomized NO_OFFER control group. The same allocator is used by the held-out backtest.
Optimality under model scores is not proof of realized profit; see `heldout_policy_comparison.csv`.


## Load data + constants

In [1]:
import sys, math, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd
from src.common import TABLES, load_processed
from src.policy import ACTIONS, COMMERCIAL, get_holdout, optimize_policy, measurement_assignment
pd.set_option("display.width", 200)
vals = pd.read_csv(TABLES / "july_customer_action_values.csv", parse_dates=["policy_as_of_ts"])
jf = load_processed("july_features")
BUDGET, MAX_OFFER_SHARE, MAX_PCT20_SHARE, HOLDOUT_SHARE = 62_000, 0.60, 0.15, 0.05
custs = np.sort(vals.customer_id.unique())
N = len(custs)
MAX_OFFERS, N_HOLD = math.floor(MAX_OFFER_SHARE * N), math.ceil(HOLDOUT_SHARE * N)
assert len(vals) == 4 * N
print(f"N={N} | max offers={MAX_OFFERS} | holdout={N_HOLD} | budget=${BUDGET:,}")


N=2776 | max offers=1665 | holdout=139 | budget=$62,000


## Reserve the random NO_OFFER measurement holdout (before optimization, independent of model scores)

In [2]:
hold_ids = get_holdout(custs, share=HOLDOUT_SHARE, seed=42)
hold = pd.DataFrame({"customer_id": hold_ids, "measurement_holdout": 1})
hold.to_csv(TABLES / "measurement_holdout.csv", index=False)
bal_cols = ["customer_tenure_days", "is_member", "prior_order_count", "days_since_last_order", "sessions_30d", "days_since_last_session"]
b = jf[["customer_id"] + bal_cols].merge(vals.query("action == 'NO_OFFER'")[["customer_id", "predicted_purchase_probability"]], on="customer_id")
b["holdout"] = b.customer_id.isin(hold_ids)
cols = bal_cols + ["predicted_purchase_probability"]
balance = pd.DataFrame({"holdout_mean": b.loc[b.holdout, cols].mean(), "rest_mean": b.loc[~b.holdout, cols].mean(), "all_mean": b[cols].mean()})
balance["std_mean_diff"] = (balance.holdout_mean - balance.rest_mean) / b[cols].std()
balance.rename_axis("variable").to_csv(TABLES / "measurement_holdout_balance.csv")
balance.round(3)

,holdout_mean,rest_mean,all_mean,std_mean_diff
customer_tenure_days,751.561,728.623,729.772,0.073
is_member,0.245,0.273,0.272,-0.065
prior_order_count,0.173,0.223,0.221,-0.099
days_since_last_order,231.348,250.996,250.303,-0.351
sessions_30d,0.086,0.099,0.099,-0.039
days_since_last_session,146.838,134.829,135.444,0.122
predicted_purchase_probability,0.124,0.129,0.129,-0.085


## Solve the same allocator used in historical evaluation
Two-stage optimization keeps the economic objective primary and resolves ties reproducibly.
The solver is SciPy's bundled HiGHS, avoiding a platform-specific CBC executable path.


In [3]:
pol = optimize_policy(vals, budget=BUDGET, max_offer_share=MAX_OFFER_SHARE,
                      max_pct20_share=MAX_PCT20_SHARE, holdout_ids=hold_ids)
status = pol.attrs["solver_status"]
objective = pol.attrs["objective"]
print("solver status:", status, "| projected objective:", round(objective, 2))
assert status == "Optimal"


solver status: Optimal | projected objective: 6408.91


## Extract the policy

In [4]:
pol.to_csv(TABLES / "optimized_policy_detailed.csv", index=False)
measurement_assignment(pol).to_csv(TABLES / "measurement_assignment.csv", index=False)
measurement_design = {
    "randomization": "ceil(5% * N) customer IDs sampled uniformly before allocation; seed 42",
    "estimand": "mean 14-day NCM in ALL POLICY-arm customers minus mean in randomized CONTROL customers",
    "policy_arm_includes_optimizer_NO_OFFER": True,
    "invalid_comparison": "offered recipients versus population control (selection-biased)",
    "analysis_function": "src.measurement.estimate_policy_itt",
    "outcomes": "same prespecified NCM window and complete follow-up for all randomized customers",
    "power": "139 controls meets the assignment minimum; adequate power is not established",
    "measurement_population": N, "control_count": len(hold_ids)}
(TABLES / "measurement_design.json").write_text(json.dumps(measurement_design, indent=2))
pol.recommended_action.value_counts().reindex(ACTIONS, fill_value=0)


recommended_action
NO_OFFER     1111
PCT_10        212
PCT_20          0
FREE_SHIP    1453
Name: count, dtype: int64

## Constraint audit + summary

In [5]:
n_off = int(pol.recommended_action.ne("NO_OFFER").sum()); n20 = int(pol.recommended_action.eq("PCT_20").sum())
spend = pol.estimated_promo_cost.sum()
all_no = vals.query("action == 'NO_OFFER'").predicted_ncm.sum()
audit = pd.DataFrame([
    ("customers", N, N, "=="), ("exactly-one-action violations", int(len(pol) != N or not pol.customer_id.is_unique), 0, "=="),
    ("expected promo spend ($)", round(spend, 2), BUDGET, "<="), ("offered customers", n_off, MAX_OFFERS, "<="),
    ("offered share", round(n_off / N, 4), MAX_OFFER_SHARE, "<="), ("PCT_20 customers", n20, None, "info"),
    ("PCT_20 share among offered", round(n20 / n_off, 4) if n_off else 0.0, MAX_PCT20_SHARE, "<="),
    ("reserved holdout customers", N_HOLD, math.ceil(HOLDOUT_SHARE * N), ">="), ("reserved holdout share", round(N_HOLD / N, 4), HOLDOUT_SHARE, ">="),
    ("holdout customers assigned NO_OFFER", int(pol.query("measurement_holdout == 1").recommended_action.eq("NO_OFFER").sum()), N_HOLD, "=="),
    ("solver status", status, "Optimal", "==")], columns=["check", "value", "limit", "rule"])
ops = {"==": lambda a, b: a == b, "<=": lambda a, b: a <= b + 1e-9, ">=": lambda a, b: a >= b - 1e-9}
audit["pass"] = [True if r == "info" else ops[r](val, lim) for val, lim, r in zip(audit.value, audit.limit, audit.rule)]
audit.to_csv(TABLES / "optimization_constraint_check.csv", index=False)
assert audit["pass"].all(), audit[~audit["pass"]]
summary = pd.DataFrame([{"solver_status": status, "objective_total_expected_incremental_ncm": round(objective, 2),
                         "total_expected_ncm_optimized": round(pol.predicted_ncm.sum(), 2), "total_expected_ncm_all_no_offer": round(all_no, 2),
                         "expected_promo_spend": round(spend, 2), "offered_customers": n_off, "offered_share": round(n_off / N, 4),
                         "pct20_share_among_offered": round(n20 / n_off, 4) if n_off else 0.0, "holdout_customers": N_HOLD,
                         **{f"n_{a}": int((pol.recommended_action == a).sum()) for a in ACTIONS}}])
summary["evaluation_basis"] = "model_projection_not_independent_validation"
summary.to_csv(TABLES / "optimization_summary.csv", index=False)
print(audit.to_string(index=False)); summary.T

                              check    value   limit rule  pass
                          customers     2776    2776   ==  True
      exactly-one-action violations        0       0   ==  True
           expected promo spend ($)  1306.57   62000   <=  True
                  offered customers     1665    1665   <=  True
                      offered share   0.5998     0.6   <=  True
                   PCT_20 customers        0    None info  True
         PCT_20 share among offered      0.0    0.15   <=  True
         reserved holdout customers      139     139   >=  True
             reserved holdout share   0.0501    0.05   >=  True
holdout customers assigned NO_OFFER      139     139   ==  True
                      solver status  Optimal Optimal   ==  True


,0
solver_status,Optimal
objective_total_expected_incremental_ncm,6408.91
total_expected_ncm_optimized,51364.2
total_expected_ncm_all_no_offer,44955.29
expected_promo_spend,1306.57
offered_customers,1665
offered_share,0.5998
pct20_share_among_offered,0.0
holdout_customers,139
n_NO_OFFER,1111
